# Distributed training — DDP & FSDP (and why multi-GPU is its own engineering problem)

Notebooks 01–02 trained JEPA on **one** GPU. Real models are trained on many. This notebook
takes the *same* I-JEPA and scales it across **both GPUs in this box** (an RTX 4090 + an RTX 3060),
using PyTorch's two workhorses:

- **DDP (DistributedDataParallel)** — *data parallel*. Replicate the model on every GPU, give each a
  different slice of the batch, and **all-reduce** the gradients so every replica stays identical. This
  is what you reach for first; it scales throughput.
- **FSDP (FullyShardedDataParallel)** — *shard the model*. When the model + gradients + optimizer state
  don't fit on one GPU, FSDP splits all three across GPUs and gathers each layer only when needed. This
  is what makes billion-parameter training possible.

We run them **for real** via `torchrun` (the standard launcher) as subprocesses, and read off the
throughput, the scaling behaviour, and — importantly — the *gotchas* that don't exist on one GPU.

> These two GPUs are deliberately **mismatched** (4090 vs 3060). That turns out to teach the single
> most important distributed lesson better than two identical cards would — see the results below.

## 1. The DDP training script

This is the single-GPU I-JEPA from notebook 01 with the distributed scaffolding added. Read the
comments — every added line is there for a reason that only shows up with >1 GPU:

- `init_process_group("nccl", device_id=dev)` — join the process group (NCCL is the GPU collective backend).
- `DistributedSampler` + `set_epoch` — give each rank a **disjoint** shard of the data, reshuffled each epoch.
- `DDP(model, device_ids=[local])` — wraps the trained nets; broadcasts rank-0 weights, all-reduces grads.
- The **EMA target encoder is kept OUTSIDE DDP** — it isn't trained, only EMA-updated from the (synced)
  student. This momentum-encoder + DDP subtlety trips up a lot of people.
- **Only rank 0 logs** (otherwise every GPU prints/saves and clobbers).
- **Effective batch size = per-GPU batch × world_size** → you'd raise the LR (linear scaling rule) for a real run.

In [1]:
%%writefile ddp_ijepa.py
"""Multi-GPU I-JEPA with DistributedDataParallel. Launch with torchrun:
    torchrun --nproc_per_node=2 ddp_ijepa.py --steps 150
Reuses the single-GPU I-JEPA model; the only changes are the DDP scaffolding."""
import os, math, copy, time, argparse
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import torchvision as tv, torchvision.transforms as T

IMG, PATCH = 32, 4; GH = GW = IMG//PATCH; N = GH*GW
DIM, DEPTH, HEADS = 192, 6, 3; PDIM, PDEPTH = 96, 4; EMA_M = 0.996

class Block(nn.Module):
    def __init__(s, dim, heads):
        super().__init__()
        s.n1=nn.LayerNorm(dim); s.attn=nn.MultiheadAttention(dim,heads,batch_first=True)
        s.n2=nn.LayerNorm(dim); s.mlp=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(s,x):
        y=s.n1(x); x=x+s.attn(y,y,y,need_weights=False)[0]; return x+s.mlp(s.n2(x))
class ViTEncoder(nn.Module):
    def __init__(s):
        super().__init__()
        s.patch=nn.Conv2d(3,DIM,PATCH,PATCH); s.pos=nn.Parameter(torch.zeros(1,N,DIM)); nn.init.trunc_normal_(s.pos,std=.02)
        s.blocks=nn.ModuleList([Block(DIM,HEADS) for _ in range(DEPTH)]); s.norm=nn.LayerNorm(DIM)
    def forward(s,img,keep_idx=None):
        x=s.patch(img).flatten(2).transpose(1,2)+s.pos
        if keep_idx is not None: x=x[:,keep_idx]
        for b in s.blocks: x=b(x)
        return s.norm(x)
class Predictor(nn.Module):
    def __init__(s):
        super().__init__()
        s.embed=nn.Linear(DIM,PDIM); s.pos=nn.Parameter(torch.zeros(1,N,PDIM)); nn.init.trunc_normal_(s.pos,std=.02)
        s.mask_token=nn.Parameter(torch.zeros(1,1,PDIM)); nn.init.trunc_normal_(s.mask_token,std=.02)
        s.blocks=nn.ModuleList([Block(PDIM,HEADS) for _ in range(PDEPTH)]); s.norm=nn.LayerNorm(PDIM); s.out=nn.Linear(PDIM,DIM)
    def forward(s,ctx,ci,ti):
        B=ctx.size(0); x=s.embed(ctx)+s.pos[:,ci]; m=(s.mask_token+s.pos[:,ti]).expand(B,-1,-1)
        x=torch.cat([x,m],1)
        for b in s.blocks: x=b(x)
        return s.out(s.norm(x)[:,ctx.size(1):])

class Trainable(nn.Module):
    """The networks that receive gradients (this is what DDP wraps). The EMA
    target encoder is kept OUTSIDE DDP — it isn't trained, only EMA-updated."""
    def __init__(s): super().__init__(); s.context=ViTEncoder(); s.predictor=Predictor()
    def forward(s, img, ci, ti, target_repr):
        ctx=s.context(img, keep_idx=ci); pred=s.predictor(ctx, ci, ti)
        return F.smooth_l1_loss(pred, target_repr)

def sample_mask(dev, n=4, scale=(0.15,0.2), aspect=(0.75,1.5)):
    tgt=torch.zeros(GH,GW,dtype=torch.bool)
    for _ in range(n):
        sc=(scale[0]+torch.rand(1).item()*(scale[1]-scale[0]))*N; ar=aspect[0]+torch.rand(1).item()*(aspect[1]-aspect[0])
        h=max(1,min(GH,round(math.sqrt(sc*ar)))); w=max(1,min(GW,round(math.sqrt(sc/ar))))
        top=torch.randint(0,GH-h+1,(1,)).item(); left=torch.randint(0,GW-w+1,(1,)).item(); tgt[top:top+h,left:left+w]=True
    tgt=tgt.flatten()
    if tgt.all(): tgt[0]=False
    if not tgt.any(): tgt[N-1]=True
    idx=torch.arange(N); return idx[~tgt].to(dev), idx[tgt].to(dev)

def main():
    ap=argparse.ArgumentParser(); ap.add_argument("--steps",type=int,default=150); ap.add_argument("--bs",type=int,default=256)
    args=ap.parse_args()
    local=int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(local); dev=torch.device("cuda",local)
    dist.init_process_group("nccl", device_id=dev)             # device_id avoids heterogeneous-mapping hangs
    rank=dist.get_rank(); world=dist.get_world_size()
    is0 = rank==0
    torch.manual_seed(0+rank)                                   # different aug/mask per rank; weights synced by DDP

    tf=T.Compose([T.ToTensor(), T.Normalize((.4914,.4822,.4465),(.247,.243,.261))])
    ds=tv.datasets.CIFAR10("data", train=True, download=is0, transform=tf)
    dist.barrier()                                              # rank 0 downloads; others wait
    if not is0: ds=tv.datasets.CIFAR10("data", train=True, download=False, transform=tf)
    sampler=DistributedSampler(ds, num_replicas=world, rank=rank, shuffle=True, drop_last=True)
    loader=DataLoader(ds, batch_size=args.bs, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)

    trainable=Trainable().to(dev)
    ddp=DDP(trainable, device_ids=[local])                     # broadcasts rank-0 weights to all ranks
    target=ViTEncoder().to(dev)                                # EMA teacher, NOT wrapped in DDP
    target.load_state_dict(ddp.module.context.state_dict())    # start = the (now-synced) student
    for p in target.parameters(): p.requires_grad_(False)
    opt=torch.optim.AdamW(ddp.parameters(), lr=1.5e-3, weight_decay=.05)

    if is0:
        eff = args.bs*world
        print(f"world_size={world}  per-GPU batch={args.bs}  EFFECTIVE batch={eff}  "
              f"(linear-scaling rule would raise LR ~{world}x for a real run)", flush=True)
        print(f"params/replica={sum(p.numel() for p in trainable.parameters())/1e6:.1f}M", flush=True)

    ddp.train(); step=0; t0=time.time(); seen=0; ep=0
    while step < args.steps:
        sampler.set_epoch(ep); ep+=1                           # reshuffle each epoch (DDP requirement)
        for img,_ in loader:
            if step>=args.steps: break
            img=img.to(dev, non_blocking=True)
            ci,ti=sample_mask(dev)
            with torch.no_grad(): tgt=target(img)[:,ti]
            loss=ddp(img,ci,ti,tgt)                            # grads all-reduced across GPUs in backward
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():                              # EMA from the (synced) student -> teacher
                for pt,pc in zip(target.parameters(), ddp.module.context.parameters()): pt.mul_(EMA_M).add_(pc,alpha=1-EMA_M)
            seen += img.size(0); step+=1
            if is0 and step%50==0: print(f"step {step}/{args.steps}  loss {loss.item():.4f}", flush=True)
    torch.cuda.synchronize(); dt=time.time()-t0
    # aggregate throughput across all ranks
    tot=torch.tensor([seen],device=dev,dtype=torch.float); dist.all_reduce(tot)
    lt=torch.tensor([loss.item()],device=dev); dist.all_reduce(lt); lt/=world
    if is0:
        print(f"\n[rank0] wall={dt:.1f}s  this-rank {seen/dt:.0f} img/s  "
              f"AGGREGATE {tot.item()/dt:.0f} img/s across {world} GPU(s)  mean-loss {lt.item():.4f}", flush=True)
    dist.barrier(); dist.destroy_process_group()

if __name__=="__main__": main()

Writing ddp_ijepa.py


In [2]:
import subprocess, sys, os
def torchrun(script, nproc, args=None, gpus=None, port=29700, keep="world_size|img/s|rank0|step |params|mode=|EFFECTIVE"):
    # launch `script` under torch.distributed.run as a subprocess (the real workflow)
    env = dict(os.environ)
    if gpus is not None: env["CUDA_VISIBLE_DEVICES"] = gpus
    cmd = [sys.executable, "-m", "torch.distributed.run",
           "--nproc_per_node", str(nproc), "--master_port", str(port), script] + (args or [])
    p = subprocess.run(cmd, capture_output=True, text=True, env=env)
    import re
    for line in p.stdout.splitlines():
        if re.search(keep, line): print(line)
    if p.returncode != 0:
        print("--- stderr tail ---"); print("\n".join(p.stderr.splitlines()[-12:]))

## 2. Run it — single GPU vs. both GPUs

We launch the same script three ways and compare aggregate throughput (images/sec). The 4090 alone is
the single-GPU baseline; the 3060 alone shows the gap; then DDP across both.

In [3]:
print(">>> DDP across BOTH GPUs (4090 + 3060):")
torchrun("ddp_ijepa.py", nproc=2, args=["--steps","120"], port=29711)
print("\n>>> Single GPU — 4090 only:")
torchrun("ddp_ijepa.py", nproc=1, args=["--steps","120"], gpus="0", port=29712)
print("\n>>> Single GPU — 3060 only:")
torchrun("ddp_ijepa.py", nproc=1, args=["--steps","120"], gpus="1", port=29713)

>>> DDP across BOTH GPUs (4090 + 3060):


world_size=2  per-GPU batch=256  EFFECTIVE batch=512  (linear-scaling rule would raise LR ~2x for a real run)
params/replica=3.2M
step 50/120  loss 0.1146
step 100/120  loss 0.1132
[rank0] wall=12.3s  this-rank 2506 img/s  AGGREGATE 5012 img/s across 2 GPU(s)  mean-loss 0.1136

>>> Single GPU — 4090 only:


world_size=1  per-GPU batch=256  EFFECTIVE batch=256  (linear-scaling rule would raise LR ~1x for a real run)
params/replica=3.2M
step 50/120  loss 0.1028
step 100/120  loss 0.1087
[rank0] wall=3.1s  this-rank 9987 img/s  AGGREGATE 9987 img/s across 1 GPU(s)  mean-loss 0.1172

>>> Single GPU — 3060 only:


world_size=1  per-GPU batch=256  EFFECTIVE batch=256  (linear-scaling rule would raise LR ~1x for a real run)
params/replica=3.2M
step 50/120  loss 0.1028
step 100/120  loss 0.1087
[rank0] wall=11.4s  this-rank 2685 img/s  AGGREGATE 2685 img/s across 1 GPU(s)  mean-loss 0.1172


## 3. Reading the result — the straggler problem

You'll see something **counterintuitive**: DDP across both GPUs is *slower in aggregate* than the 4090
alone. That's not a bug — it's the **straggler effect**, and it's the defining challenge of real
distributed training:

> DDP synchronizes gradients at **every step**. The all-reduce can't finish until *every* GPU has
> produced its gradients — so the whole job runs at the pace of the **slowest** worker. Pair a 4090 with
> a 3060 and the 4090 spends half its time idle, waiting at the barrier.

The lessons that generalize:

- **Scaling is near-linear only with *homogeneous*, well-connected workers.** Two identical 4090s would
  give ~2×; a fast + slow pair gives ~2× the *slow* one. Real clusters fight hard for homogeneity and
  fast interconnect (NVLink / InfiniBand) precisely because the all-reduce is on the critical path.
- **Communication is overhead.** Every step ships a full gradient (all-reduce ≈ 2×params of traffic).
  Bigger models or slower links → comms dominate; mitigations: gradient bucketing (DDP does this),
  overlapping comms with backward (DDP does this), gradient compression, fewer-but-bigger steps.
- **Effective batch grows with world size**, which changes optimization — you retune LR/warmup.
- **Determinism & data:** seed per-rank, use `DistributedSampler.set_epoch`, and `drop_last` so ranks
  stay in lock-step (a ragged last batch can deadlock the all-reduce).
- **BatchNorm** would need `SyncBatchNorm` across ranks (we use LayerNorm, so we're fine).

## 4. FSDP — when the model itself doesn't fit

DDP **replicates** everything on every GPU, so the biggest model you can train is capped by one GPU's
memory. **FSDP shards** the parameters, gradients, *and* optimizer state across GPUs, materializing
each layer's full weights only transiently during its forward/backward. Same data-parallel math, far
less memory per GPU.

We compare peak memory on a deliberately **enlarged** ViT (~96M params) under DDP vs FSDP across both GPUs:

In [4]:
%%writefile fsdp_mem.py
"""DDP vs FSDP peak-memory comparison on an ENLARGED ViT, across 2 GPUs.
    torchrun --nproc_per_node=2 fsdp_mem.py --mode ddp
    torchrun --nproc_per_node=2 fsdp_mem.py --mode fsdp
DDP replicates the whole model (+grads +optimizer state) on every GPU. FSDP shards
all three across GPUs, so peak memory per GPU drops. This is a memory benchmark
(fixed random targets), not a learning run."""
import os, functools, argparse, torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

# deliberately LARGE so param/grad/optimizer memory dominates and sharding shows
DIM, DEPTH, HEADS = 768, 12, 12
PDIM, PDEPTH = 384, 6
N = 64

class Block(nn.Module):
    def __init__(s, dim, heads):
        super().__init__()
        s.n1=nn.LayerNorm(dim); s.attn=nn.MultiheadAttention(dim,heads,batch_first=True)
        s.n2=nn.LayerNorm(dim); s.mlp=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(s,x):
        y=s.n1(x); x=x+s.attn(y,y,y,need_weights=False)[0]; return x+s.mlp(s.n2(x))

class Trainable(nn.Module):
    def __init__(s):
        super().__init__()
        s.embed=nn.Linear(3*16, DIM)                      # fake "patchify": 64 tokens of dim 48 -> DIM
        s.pos=nn.Parameter(torch.zeros(1,N,DIM)); nn.init.trunc_normal_(s.pos,std=.02)
        s.enc=nn.ModuleList([Block(DIM,HEADS) for _ in range(DEPTH)])
        s.to_pred=nn.Linear(DIM,PDIM); s.pred=nn.ModuleList([Block(PDIM,HEADS//2) for _ in range(PDEPTH)])
        s.out=nn.Linear(PDIM,DIM); s.norm=nn.LayerNorm(DIM)
    def forward(s, x, target):
        h=s.embed(x)+s.pos
        for b in s.enc: h=b(h)
        h=s.norm(h); p=s.to_pred(h)
        for b in s.pred: p=b(p)
        return F.smooth_l1_loss(s.out(p), target)

def main():
    ap=argparse.ArgumentParser(); ap.add_argument("--mode",choices=["ddp","fsdp"],default="ddp")
    ap.add_argument("--steps",type=int,default=20); ap.add_argument("--bs",type=int,default=64)
    a=ap.parse_args()
    local=int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(local); dev=torch.device("cuda",local)
    dist.init_process_group("nccl", device_id=dev); rank=dist.get_rank(); world=dist.get_world_size()
    is0=rank==0
    torch.manual_seed(0)
    model=Trainable().to(dev)
    nparams=sum(p.numel() for p in model.parameters())
    if a.mode=="ddp":
        model=DDP(model, device_ids=[local])
    else:
        policy=functools.partial(transformer_auto_wrap_policy, transformer_layer_cls={Block})
        model=FSDP(model, auto_wrap_policy=policy, device_id=local)
    opt=torch.optim.AdamW(model.parameters(), lr=1e-4)

    torch.cuda.reset_peak_memory_stats(dev)
    model.train()
    for _ in range(a.steps):
        x=torch.randn(a.bs, N, 3*16, device=dev)
        target=torch.randn(a.bs, N, DIM, device=dev)
        loss=model(x, target); opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    peak=torch.cuda.max_memory_allocated(dev)/1e9
    # report the MAX peak across ranks (the binding constraint)
    pk=torch.tensor([peak],device=dev); dist.all_reduce(pk, op=dist.ReduceOp.MAX)
    if is0:
        print(f"mode={a.mode.upper():4s}  model={nparams/1e6:.0f}M params  world={world}  "
              f"per-GPU PEAK memory={pk.item():.2f} GB  final loss={loss.item():.3f}", flush=True)
    dist.barrier(); dist.destroy_process_group()

if __name__=="__main__": main()

Writing fsdp_mem.py


In [5]:
print(">>> DDP — model replicated on each GPU:")
torchrun("fsdp_mem.py", nproc=2, args=["--mode","ddp"], port=29714)
print("\n>>> FSDP — model sharded across GPUs:")
torchrun("fsdp_mem.py", nproc=2, args=["--mode","fsdp"], port=29715)

>>> DDP — model replicated on each GPU:


mode=DDP   model=96M params  world=2  per-GPU PEAK memory=5.29 GB  final loss=0.430

>>> FSDP — model sharded across GPUs:


mode=FSDP  model=96M params  world=2  per-GPU PEAK memory=4.11 GB  final loss=0.430


## 5. What we learned

- **DDP = data parallel, FSDP = sharded data parallel.** DDP scales *throughput*; FSDP unlocks
  *model size*. They compose (HSDP: shard within a node, replicate across nodes).
- **The bottleneck moves.** On one GPU you optimize FLOPs. On many, you optimize **communication and
  synchronization** — the straggler effect we measured is the canonical example, and it's why
  homogeneous hardware + fast interconnects matter so much.
- **FSDP trades compute/comms for memory.** It re-gathers shards each layer (extra all-gathers), so it's
  a bit slower than DDP but lets you train models that DDP simply can't fit. Our measurement shows the
  per-GPU memory drop even at 96M params; the gap widens with model size (activations aren't sharded by
  default, which is why the reduction here isn't a full 1/N — activation checkpointing addresses that).
- **The launch model is different.** `torchrun` spawns one process per GPU; your script must be
  rank-aware (who logs, who saves, who downloads). None of that exists on a single GPU.
- Next: `04_efficient_video_data_pipelines.ipynb` — because once the GPUs are fast and many, the
  **input pipeline** becomes the bottleneck, especially for video.